# Lyrics Transcribe in Kaggle

**Pipeline:** anvuew BS-Roformer (vocal isolation) → WhisperX or GigaAM (ASR with word timestamps) → standalone karaoke HTML page. Optional: bring your own `lyrics.txt` and use MMS forced alignment instead of ASR.

## Before you run

In the right-side panel:
- **Settings → Accelerator** = `GPU T4 x2` (or `P100` / `L4`).
- **Settings → Internet** = `On` (required to clone the repo and download model weights).
- **Settings → Persistence** = `Files only` (or off — not needed).

## How to provide your audio

Two options:
1. **Upload via the Input panel** → click `+ Add Input` → `Upload` → drop your audio. It will appear at `/kaggle/input/<dataset-name>/`. Set `INPUT_DIR` below to that path.
2. **Drag-drop into the file browser** (left side, under `kaggle/working`). Set `INPUT_DIR = '/kaggle/working'`.

Optionally drop a `*.txt` with the exact lyrics next to your audio — the run cell auto-switches to forced alignment when it finds one.

## 1. Setup — clone, install, env, download anvuew checkpoint (~196 MB)

One-shot init: clones the repo into `/kaggle/working/LyricsTranscribe`, creates a `.venv`, sets the env vars faster-whisper/gigaam need, and pulls the anvuew BS-Roformer checkpoint from the GitHub release.

WhisperX (~3 GB) and GigaAM (~430 MB) auto-download into `models/` on first ASR call. MMS aligner (~1.2 GB) auto-downloads on first `--lyrics` call.

In [ ]:
# 1) clone repo into the writable working dir
!rm -rf /kaggle/working/LyricsTranscribe
!git clone https://github.com/Beatloo-Labs/LyricsTranscribe.git /kaggle/working/LyricsTranscribe
%cd /kaggle/working/LyricsTranscribe

# 2) venv + deps (torch from pytorch index for the CUDA build)
!pip install -q uv
!uv venv
!uv pip install -p .venv/bin/python --index-url https://download.pytorch.org/whl/cu128 torch==2.8.0 torchaudio==2.8.0
!uv pip install -p .venv/bin/python -r requirements.txt

# 3) env: cuDNN libs (from the nvidia-cudnn pip wheel) + system NVIDIA driver
import os, glob
cudnn = glob.glob('/kaggle/working/LyricsTranscribe/.venv/lib/python*/site-packages/nvidia/cudnn/lib/')
if not cudnn:
    raise RuntimeError('cuDNN libs not found — torch install must have failed above')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = 'true'
os.environ['LD_LIBRARY_PATH'] = cudnn[0] + ':/usr/lib64-nvidia:' + os.environ.get('LD_LIBRARY_PATH', '')

# 4) anvuew checkpoint
!mkdir -p models/anvuew
!wget -q --show-progress \
  https://github.com/Beatloo-Labs/LyricsTranscribe/releases/download/weights-v1/bs_roformer_ft1_anvuew_sdr_12.55.ckpt \
  -O models/anvuew/bs_roformer_ft1_anvuew_sdr_12.55.ckpt

## 2. Pick the audio (and optional lyrics) from your input

Set `INPUT_DIR` to where your files sit. Defaults below cover the two common cases — uncomment whichever you used.

In [ ]:
import os, glob

# --- pick ONE of these ---
INPUT_DIR = '/kaggle/working'         # if you drag-dropped files into the working tree
# INPUT_DIR = '/kaggle/input/your-dataset-name'   # if you attached a dataset

AUDIO_EXTS = ('.mp3', '.wav', '.flac', '.m4a', '.ogg', '.opus', '.aac', '.aiff', '.aif')
AUDIO_FILES = sorted(p for p in glob.glob(f'{INPUT_DIR}/**/*', recursive=True)
                     if p.lower().endswith(AUDIO_EXTS))
TXT_FILES = sorted(glob.glob(f'{INPUT_DIR}/**/*.txt', recursive=True))
LYRICS_FILE = TXT_FILES[0] if TXT_FILES else None

print(f'audio:  {AUDIO_FILES}')
print(f'lyrics: {LYRICS_FILE or "(none — will use ASR)"}')
if not AUDIO_FILES:
    raise SystemExit(f'no audio files in {INPUT_DIR}. Drop a file there or set INPUT_DIR.')

## 3. Run the CLI

Pick `--model whisperx` (multilingual) or `--model gigaam` (Russian-tuned). If a `lyrics.txt` was found in step 2, the cell auto-switches to MMS forced alignment and `--model` is ignored. First run downloads the chosen model (~3 GB / ~430 MB / ~1.2 GB) into `models/`.

In [ ]:
import subprocess

MODEL    = 'whisperx'   # 'whisperx' or 'gigaam' (ignored if LYRICS_FILE is set)
LANGUAGE = 'ru'         # ISO code, or empty string for auto-detect
ISOLATE  = True         # run anvuew vocal separation before ASR/align
OUT_DIR  = '/kaggle/working/cli-out'

args = ['.venv/bin/python', 'transcribe.py',
        '--language', LANGUAGE, '--out', OUT_DIR]
if not ISOLATE:
    args.append('--no-isolate')

if LYRICS_FILE:
    args += ['--lyrics', LYRICS_FILE]
    print(f'[align] using lyrics: {LYRICS_FILE}')
else:
    args += ['--model', MODEL]
    print(f'[asr] using model: {MODEL}')

args += AUDIO_FILES
subprocess.run(args, env=os.environ, check=True)

## 4. Inspect / package the output

Each track lands in `cli-out/<name>__<engine>__<timestamp>/` with the audio, JSON microformat, and a standalone karaoke HTML page. Kaggle's right-pane file browser lets you click any of them to download individually. The cell below also bundles everything into a single zip in `/kaggle/working/karaoke-output.zip` — that file shows up in **Output** at the end of the run for one-click download.

In [ ]:
!ls -la /kaggle/working/cli-out/
!cd /kaggle/working/cli-out && zip -rq /kaggle/working/karaoke-output.zip .

## (Optional) Run the FastAPI server

Kaggle doesn't expose arbitrary ports to the public web (no `serve_kernel_port_as_window` like Colab). The server still starts — you can hit it from inside the notebook for testing, but external access requires a tunnel like `cloudflared` or `ngrok` set up manually.

In [ ]:
import subprocess, time
proc = subprocess.Popen(['.venv/bin/python', 'server.py'], env=os.environ)
time.sleep(4)
import urllib.request
print(urllib.request.urlopen('http://127.0.0.1:8000/health').read().decode())